In [21]:
import pandas as pd
from sqlalchemy import create_engine, text
import sys
sys.path.append("..") # con_lib is the parent directory of this notebook
from con_lib1 import connection_string
sys.path.append("../src")
import db

engine = create_engine(connection_string)
db.set_engine(engine)

In [22]:
from sqlalchemy import create_engine
from con_lib1 import connection_string

engine = create_engine(connection_string)

with engine.connect() as connection:
    print("Connection successful!")

Connection successful!


In [23]:
pd.read_sql('friends', con=engine)

,friends_id,friends_name,phone_number,email,max_loans
0,1,Aarav Sharma,4915112345678,aarav.sharma@example.com,5
1,2,Priya Patel,4915223456789,priya.patel@example.com,3
2,3,Daniel Weber,4916334567890,daniel.weber@example.com,5
3,4,Emma Fischer,4917445678901,emma.fischer@example.com,4
4,5,Rahul Mehta,4917556789012,rahul.mehta@example.com,3


In [24]:
pd.read_sql('books', con=engine)

,isbn,title,author,genre,book_language,is_available
0,9780061120084,To Kill a Mockingbird,Harper Lee,Classic,English,0
1,9780132350884,Clean Code,Robert C. Martin,Programming,English,1
2,9780140449136,The Alchemist,Paulo Coelho,Fiction,English,1
3,9780141185064,Animal Farm,George Orwell,Political Fiction,English,1
4,9780143454212,The Midnight Library,Matt Haig,Fiction,English,0
5,9780261102217,The Hobbit,J.R.R. Tolkien,Fantasy,English,1
6,9780451524935,1984,George Orwell,Dystopian,English,0
7,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Classic,English,1
8,9781491957660,Python Crash Course,Eric Matthes,Programming,English,1
9,9788184001754,The Palace of Illusions,Chitra Banerjee Divakaruni,Historical Fiction,English,1


In [25]:
pd.read_sql('loans', con=engine)

,loan_id,isbn,friends_id,loan_date,due_date,extended_due_date,extension_count,return_date
0,1,9780061120084,1,2026-09-10,2026-09-24,NaT,0,NaT
1,2,9780451524935,2,2026-09-11,2026-09-25,NaT,0,NaT
2,3,9780143454212,3,2026-09-12,2026-09-26,NaT,0,NaT


In [33]:



# ---------------- BOOKS ---------------- #

def create_book(isbn, title, author, genre, book_language):
    engine = db.get_engine()

    query = """
    INSERT INTO books
    (isbn, title, author, genre, book_language)
    VALUES (%s, %s, %s, %s, %s)
    """

    with engine.connect() as connection:
        connection.exec_driver_sql(
            query,
            (isbn, title, author, genre, book_language)
        )
        connection.commit()


# ---------------- FRIENDS ---------------- #

def create_friend(friends_name, phone_number, email, max_loans):
    engine = db.get_engine()

    query = """
    INSERT INTO friends
    (friends_name, phone_number, email, max_loans)
    VALUES (%s, %s, %s, %s)
    """

    with engine.connect() as connection:
        connection.exec_driver_sql(
            query,
            (friends_name, phone_number, email, max_loans)
        )
        connection.commit()

In [34]:
create_book(
    "9781234567890",
    "My New Book",
    "John Smith",
    "Fiction",
    "English"
)

In [37]:
pd.read_sql('books', con=engine)

,isbn,title,author,genre,book_language,is_available
0,9780061120084,To Kill a Mockingbird,Harper Lee,Classic,English,0
1,9780132350884,Clean Code,Robert C. Martin,Programming,English,1
2,9780140449136,The Alchemist,Paulo Coelho,Fiction,English,1
3,9780141185064,Animal Farm,George Orwell,Political Fiction,English,1
4,9780143454212,The Midnight Library,Matt Haig,Fiction,English,0
5,9780261102217,The Hobbit,J.R.R. Tolkien,Fantasy,English,1
6,9780451524935,1984,George Orwell,Dystopian,English,0
7,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Classic,English,1
8,9781234567890,My New Book,John Smith,Fiction,English,1
9,9781491957660,Python Crash Course,Eric Matthes,Programming,English,1


In [38]:
pd.read_sql('friends', con=engine)

,friends_id,friends_name,phone_number,email,max_loans
0,1,Aarav Sharma,4915112345678,aarav.sharma@example.com,5
1,2,Priya Patel,4915223456789,priya.patel@example.com,3
2,3,Daniel Weber,4916334567890,daniel.weber@example.com,5
3,4,Emma Fischer,4917445678901,emma.fischer@example.com,4
4,5,Rahul Mehta,4917556789012,rahul.mehta@example.com,3
5,11,Sayali,491234567890,sayali@example.com,5


In [40]:
# ---------------- LOANS ---------------- #

def create_loan(isbn, friends_id, due_date):

    engine = db.get_engine()

    df = pd.DataFrame(
        [[isbn, friends_id, due_date]],
        columns=[
            "isbn",
            "friends_id",
            "due_date"
        ]
    )

    df.to_sql(
        "loans",
        if_exists="append",
        con=engine,
        index=False
    )

    message = f"Loan created for book '{isbn}'."

    return message

In [35]:
create_friend(
    "Sayali",
    "491234567890",
    "sayali@example.com",
    5
)

In [41]:
create_loan(
    "9781234567890",
    1,
    "2026-09-30"
)

"Loan created for book '9781234567890'."

In [42]:
pd.read_sql('loans', con=engine)

,loan_id,isbn,friends_id,loan_date,due_date,extended_due_date,extension_count,return_date
0,1,9780061120084,1,2026-09-10,2026-09-24,NaT,0,NaT
1,2,9780451524935,2,2026-09-11,2026-09-25,NaT,0,NaT
2,3,9780143454212,3,2026-09-12,2026-09-26,NaT,0,NaT
3,4,9781234567890,1,2026-09-15,2026-09-30,NaT,0,NaT


In [43]:



def update_friend(friend_id, field, new_data):

    engine = db.get_engine()

    query = text(f"""
        UPDATE friends
        SET {field} = :new_data
        WHERE friends_id = :friend_id
    """)

    with engine.begin() as connection:
        connection.execute(
            query,
            {
                "new_data": new_data,
                "friend_id": friend_id
            }
        )

    return "Friend updated."


def update_book(isbn, field, new_data):

    engine = db.get_engine()

    query = text(f"""
        UPDATE books
        SET {field} = :new_data
        WHERE isbn = :isbn
    """)

    with engine.begin() as connection:
        connection.execute(
            query,
            {
                "new_data": new_data,
                "isbn": isbn
            }
        )

    return "Book updated."


def update_loan(loan_id, field, new_data):

    engine = db.get_engine()

    query = text(f"""
        UPDATE loans
        SET {field} = :new_data
        WHERE loan_id = :loan_id
    """)

    with engine.begin() as connection:
        connection.execute(
            query,
            {
                "new_data": new_data,
                "loan_id": loan_id
            }
        )

    return "Loan updated."

In [44]:
update_friend(2, "friends_name", "Priya")

'Friend updated.'

In [45]:
pd.read_sql('friends', con=engine)

,friends_id,friends_name,phone_number,email,max_loans
0,1,Aarav Sharma,4915112345678,aarav.sharma@example.com,5
1,2,Priya,4915223456789,priya.patel@example.com,3
2,3,Daniel Weber,4916334567890,daniel.weber@example.com,5
3,4,Emma Fischer,4917445678901,emma.fischer@example.com,4
4,5,Rahul Mehta,4917556789012,rahul.mehta@example.com,3
5,11,Sayali,491234567890,sayali@example.com,5
